# 4주차 B3 · UCI HAR Inception-ResNet 1D + CBAM

B1의 Inception-ResNet 1D에 **CBAM(채널 attention → 시간 attention)**을 하나만 추가합니다. B2에서 SE가 있던 자리에 SE 대신 CBAM을 넣었습니다.

이번 실험의 핵심 질문은 다음과 같습니다.

> 같은 Inception-ResNet 구조에서 "어떤 특징 채널이 중요한가"(SE)에 더해 "어느 시점이 중요한가"까지 조절하면 더 좋은 UCI HAR 모델이 되는가?

핵심 비교는 두 가지입니다.

| 비교 | 달라지는 것 | 알 수 있는 것 |
|---|---|---|
| `B3 − B1` | CBAM 추가 | Inception-ResNet에 채널+시간 attention을 더한 효과 |
| `B3 − B2` | SE → CBAM | 채널 attention만 쓸 때와 시간 attention까지 쓸 때의 차이 |

데이터 분할, 블록 수, kernel, 학습 조건, residual scale, attention 위치는 B1·B2와 같습니다.

**실행 상태:** 이 파일은 Colab 실행 전 실험 설계본입니다. 저장된 점수를 실제 결과로 간주하지 않습니다.

**중요:** 모델 선택은 피험자 단위 validation으로 수행하고, 공식 test는 최종 구조를 정한 뒤 한 번만 확인합니다.


## 먼저 확인할 가설: 시간 attention이 UCI HAR에서 도움이 되는가?

CBAM은 두 단계로 특징을 재가중합니다.

1. **채널 attention:** SE와 비슷하게 특징 채널마다 0~1 중요도를 곱합니다. 단, 시간 평균(avg)뿐 아니라 시간 최댓값(max)도 함께 사용합니다.
2. **시간 attention:** 128(또는 64)개 시점마다 0~1 중요도를 곱해 "언제"가 중요한지 조절합니다.

가설은 다음과 같습니다.

- **동적 행동(걷기·계단):** 윈도우 안에서 발이 닿는 순간처럼 특정 시점이 두드러지므로 시간 attention이 도움이 될 수 있습니다.
- **정적 자세(SIT/STAND/LAYING):** 윈도우 전체가 거의 비슷한 신호이므로 **강조할 특정 시점이 없어 시간 attention의 이득은 작을 것**으로 예상합니다.

현재 검증 오류의 대부분은 SIT↔STAND입니다. 따라서 B3가 전체 정확도를 크게 올릴 것이라고 미리 가정하지 않습니다. 동적 오류가 줄었는지, SIT/STAND는 그대로인지를 나눠서 확인합니다.

CBAM도 SE처럼 휴대폰 방향을 직접 보정하지 않습니다. 원본 센서축이 아니라 학습된 특징 채널과 시점을 재가중합니다.


## 0. B3 구조와 비교 원칙

B3는 B2에서 SE가 있던 자리(residual branch, `Add` 이전)에 CBAM을 넣습니다.

```text
입력 x
 ├─ bottleneck 1×1 → Conv1D(3)
 ├─ bottleneck 1×1 → Conv1D(7)
 ├─ bottleneck 1×1 → Conv1D(15)
 └─ MaxPool(3)      → Conv1D(1)
             ↓ Concatenate
          projection 1×1
             ↓ BatchNormalization
          CBAM: 채널 attention → 시간 attention
             ↓ residual scaling α=0.2
          Add([shortcut, residual])
             ↓ ReLU
```

SE와 같은 위치에 두어야 `B3 − B2`를 "SE 대신 CBAM을 쓴 효과"로 해석할 수 있습니다. shortcut은 그대로 보존합니다.

### B1·B2·B3에서 고정하는 조건

| 항목 | B1 | B2 | B3 |
|---|---|---|---|
| Inception kernel | 3·7·15 | 3·7·15 | 3·7·15 |
| 블록 | 32채널 2개 + 64채널 2개 | 동일 | 동일 |
| Residual / scale | 사용 / 0.2 | 동일 | 동일 |
| Attention | 없음 | SE (ratio 8) | **CBAM (ratio 8, 시간 kernel 7)** |
| Attention 위치 | – | BN 뒤, scale 전 | BN 뒤, scale 전 |
| 학습·검증 분리와 학습 조건 | 동일 | 동일 | 동일 |

CBAM의 채널 attention MLP 크기(ratio 8)를 SE와 같게 두었으므로, B2와 B3의 파라미터 차이는 시간 attention의 Conv1D(7×2=14개)와 Dense bias 정도로 작습니다. 따라서 성능 차이를 "파라미터가 늘어서"라고 보기는 어렵습니다.


## 1. 라이브러리 불러오기

In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

## 2. 실험 설정

먼저 `USE_RESIDUAL=True`, `USE_CBAM=True`를 유지하고 `SEED`만 0·1·2로 바꿉니다. 이것이 B3의 정식 결과입니다.

- `CBAM_RATIO=8`: 채널 attention의 중간 크기입니다. B2의 `SE_RATIO=8`과 같게 맞췄습니다.
- `CBAM_TEMPORAL_KERNEL=7`: 시간 attention이 한 번에 보는 범위입니다. 50Hz에서 약 0.14초이며, CBAM 원 논문의 spatial kernel 7을 1D로 옮긴 값입니다.


In [ ]:
SEED = 0
USE_RESIDUAL = True
# B3 변경: USE_SE 대신 USE_CBAM. ratio는 B2의 SE와 같은 8, 시간 attention kernel은 7
USE_CBAM = True
CBAM_RATIO = 8
CBAM_TEMPORAL_KERNEL = 7
RESIDUAL_SCALE = 0.2

np.random.seed(SEED)
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

EXPERIMENT_NAME = (
    f"B3-res{int(USE_RESIDUAL)}"
    f"-cbam{int(USE_CBAM)}"
    f"-r{CBAM_RATIO}"
    f"-k{CBAM_TEMPORAL_KERNEL}"
)
print("experiment:", EXPERIMENT_NAME, "| seed:", SEED)

## 3. Google Drive와 데이터 경로 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"
assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"

## 4. 9개 센서 신호와 데이터 로더 정의

In [ ]:
signals = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]


def load_data(split):
    channels = []

    for signal in signals:
        path = f"{DATA_PATH}/{split}/Inertial Signals/{signal}_{split}.txt"
        channels.append(np.loadtxt(path))

    # (9, N, 128)이 아니라 각 (N, 128) 배열을 마지막 축에 쌓아 (N, 128, 9)로 만듭니다.
    X = np.stack(channels, axis=-1)
    y = np.loadtxt(f"{DATA_PATH}/{split}/y_{split}.txt").astype(int) - 1
    subject = np.loadtxt(f"{DATA_PATH}/{split}/subject_{split}.txt").astype(int)

    return X, y, subject


X_train_all, y_train_all, s_train_all = load_data("train")
X_test, y_test, s_test = load_data("test")

print("official train:", X_train_all.shape)
print("official test :", X_test.shape, "(최종 셀 전까지 평가하지 않음)")

## 5. 피험자 단위 validation 분리

같은 사람의 윈도우가 학습과 validation에 동시에 들어가지 않도록 피험자 5·11·17·23·29를 통째로 분리합니다. 특히 5번과 17번의 `SITTING`/`STANDING` 성능을 핵심 지표로 봅니다.

In [ ]:
VAL_SUBJECTS = [5, 11, 17, 23, 29]
FOCUS_SUBJECTS = [5, 17]

val_mask = np.isin(s_train_all, VAL_SUBJECTS)

X_val = X_train_all[val_mask]
y_val = y_train_all[val_mask]
s_val = s_train_all[val_mask]

X_train = X_train_all[~val_mask]
y_train = y_train_all[~val_mask]
s_train = s_train_all[~val_mask]

print("train:", X_train.shape, "subjects:", sorted(np.unique(s_train)))
print("val  :", X_val.shape, "subjects:", sorted(np.unique(s_val)))

## 6. 학습 데이터 통계로 정규화

In [ ]:
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

X_train = ((X_train - mean) / (std + 1e-8)).astype("float32")
X_val = ((X_val - mean) / (std + 1e-8)).astype("float32")
X_test = ((X_test - mean) / (std + 1e-8)).astype("float32")

n_outputs = 6
y_train_onehot = to_categorical(y_train, num_classes=n_outputs)
y_val_onehot = to_categorical(y_val, num_classes=n_outputs)

## 7. Inception-residual 1D 블록과 CBAM 위치

Inception branch와 projection은 B1·B2와 동일합니다. B3의 유일한 구조 변경은 B2의 SE 자리를 CBAM으로 바꾼 것입니다: `BatchNormalization → CBAM → residual scaling → Add`.

- CBAM을 각 Inception branch에 따로 넣지 않습니다. B2와 같은 위치여야 SE와 공정하게 비교할 수 있습니다.
- CBAM을 shortcut에는 넣지 않습니다. 입력 정보가 그대로 흐르는 residual 통로를 보존합니다.
- 채널 attention과 시간 attention의 sigmoid 값은 초기에 대체로 0.5 근처일 수 있습니다. 두 번 곱해지므로 학습 초기에는 residual 신호가 B2보다 더 작아질 수 있습니다(약 0.5×0.5×0.2). 따라서 향상을 미리 가정하지 않습니다.


### 7-1. 1D CBAM 블록

입력 shape이 `(batch, time, channels)`일 때 동작은 다음과 같습니다. 코드는 `00_CNN_Blocks_Concept_Guide`의 `cbam_1d`와 같습니다.

**① 채널 attention: 무엇이 중요한가?**

1. 시간축 평균(`GlobalAveragePooling1D`)과 시간축 최댓값(`GlobalMaxPooling1D`)으로 채널마다 값 두 개를 만듭니다.
2. 두 값을 **같은 MLP**(Dense C/r → Dense C)에 통과시켜 더한 뒤 sigmoid를 적용합니다.
3. 채널 중요도를 모든 시점에 같은 값으로 곱합니다.

$$M_c(U)=\sigma(MLP(AvgPool_t(U))+MLP(MaxPool_t(U)))$$

**② 시간 attention: 언제가 중요한가?**

1. 채널 attention이 적용된 특징을 채널축으로 평균·최댓값을 내어 `(batch, time, 2)`를 만듭니다.
2. Conv1D(kernel 7) + sigmoid로 시점마다 0~1 중요도 `(batch, time, 1)`를 만듭니다.
3. 시점 중요도를 모든 채널에 같은 값으로 곱합니다.

$$M_t(U')=\sigma(f^7([AvgPool_c(U');MaxPool_c(U')]))$$

예를 들어 `(batch, 128, 32)`는 채널 attention에서 `(batch, 1, 32)` 가중치를, 시간 attention에서 `(batch, 128, 1)` 가중치를 받고 최종 출력은 `(batch, 128, 32)`로 유지됩니다.

| 구분 | SE (B2) | CBAM (B3) |
|---|---|---|
| 채널 요약 | 시간 평균 | 시간 평균 + 시간 최댓값 |
| 시간 위치 재가중 | 없음 | 있음 |
| 출력 shape | 입력과 같음 | 입력과 같음 |


In [ ]:
def cbam_1d(x, reduction_ratio=8, temporal_kernel=7, name="cbam"):
    # B3 변경: B2의 se_block_1d 대신 CBAM을 정의합니다.
    # 이유: 채널 중요도(SE와 비슷)에 더해 시점별 중요도까지 학습하는 효과를 보기 위함
    channels = x.shape[-1]
    if channels is None:
        raise ValueError("CBAM 블록은 채널 수가 확정되어 있어야 합니다.")

    channels = int(channels)
    hidden = max(channels // reduction_ratio, 1)

    # 채널 attention: 평균·최댓값 두 경로가 같은 MLP를 공유합니다.
    shared_reduce = layers.Dense(hidden, activation="relu", name=f"{name}_shared_reduce")
    shared_expand = layers.Dense(channels, name=f"{name}_shared_expand")

    avg_descriptor = layers.GlobalAveragePooling1D(name=f"{name}_channel_avg")(x)
    max_descriptor = layers.GlobalMaxPooling1D(name=f"{name}_channel_max")(x)

    avg_score = shared_expand(shared_reduce(avg_descriptor))
    max_score = shared_expand(shared_reduce(max_descriptor))
    channel_score = layers.Add(name=f"{name}_channel_add")([avg_score, max_score])
    channel_score = layers.Activation("sigmoid", name=f"{name}_channel_sigmoid")(channel_score)
    channel_score = layers.Reshape((1, channels), name=f"{name}_channel_reshape")(channel_score)
    x = layers.Multiply(name=f"{name}_channel_scale")([x, channel_score])

    # 시간 attention: 각 시점에서 채널 평균과 최댓값으로 그 시점의 중요도를 계산합니다.
    temporal_avg = layers.Lambda(
        lambda tensor: tf.reduce_mean(tensor, axis=-1, keepdims=True),
        name=f"{name}_temporal_avg",
    )(x)
    temporal_max = layers.Lambda(
        lambda tensor: tf.reduce_max(tensor, axis=-1, keepdims=True),
        name=f"{name}_temporal_max",
    )(x)
    temporal_descriptor = layers.Concatenate(
        axis=-1, name=f"{name}_temporal_concat"
    )([temporal_avg, temporal_max])
    temporal_score = layers.Conv1D(
        1, temporal_kernel, padding="same", activation="sigmoid",
        use_bias=False, name=f"{name}_temporal_conv",
    )(temporal_descriptor)

    # 시점 중요도를 모든 채널에 같은 값으로 곱합니다.
    return layers.Multiply(name=f"{name}_temporal_scale")([x, temporal_score])


In [ ]:
def inception_residual_block(
    x,
    filters,
    bottleneck_filters=16,
    kernel_sizes=(3, 7, 15),
    residual_scale=0.2,
    use_residual=True,
    use_cbam=True,
    cbam_ratio=8,
    cbam_temporal_kernel=7,
    name="ir",
):
    if filters % 4 != 0:
        raise ValueError("filters는 네 branch에 나눌 수 있도록 4의 배수여야 합니다.")

    branch_filters = filters // 4
    shortcut = x

    # 큰 kernel을 적용하기 전에 채널을 줄여 계산량을 제한합니다.
    bottleneck = layers.Conv1D(
        bottleneck_filters, 1, padding="same", use_bias=False,
        name=f"{name}_bottleneck",
    )(x)
    bottleneck = layers.BatchNormalization(name=f"{name}_bottleneck_bn")(bottleneck)
    bottleneck = layers.Activation("relu", name=f"{name}_bottleneck_relu")(bottleneck)

    branches = []
    for kernel_size in kernel_sizes:
        branch = layers.Conv1D(
            branch_filters, kernel_size, padding="same", use_bias=False,
            name=f"{name}_conv_k{kernel_size}",
        )(bottleneck)
        branches.append(branch)

    pool_branch = layers.MaxPooling1D(
        pool_size=3, strides=1, padding="same", name=f"{name}_pool"
    )(x)
    pool_branch = layers.Conv1D(
        branch_filters, 1, padding="same", use_bias=False,
        name=f"{name}_pool_projection",
    )(pool_branch)
    branches.append(pool_branch)

    merged = layers.Concatenate(axis=-1, name=f"{name}_concatenate")(branches)

    # Inception 결과를 shortcut과 같은 채널 수로 투영합니다.
    residual = layers.Conv1D(
        filters, 1, padding="same", use_bias=False,
        name=f"{name}_residual_projection",
    )(merged)
    residual = layers.BatchNormalization(name=f"{name}_residual_bn")(residual)

    # B3 변경: B2의 SE 자리에 CBAM을 넣습니다 (BN 뒤, residual scale 전).
    # 이유: 위치를 B2와 같게 두어야 B3 - B2 를 "SE 대신 CBAM" 효과로 해석할 수 있음
    #       shortcut은 그대로 두고 residual branch의 특징만 채널·시점별로 재가중합니다.
    if use_cbam:
        residual = cbam_1d(
            residual,
            reduction_ratio=cbam_ratio,
            temporal_kernel=cbam_temporal_kernel,
            name=f"{name}_cbam",
        )

    # 모든 ablation 조건에 같은 residual 배율을 적용합니다.
    # 비교 조건에서 shortcut 유무만 달라지게 합니다.
    residual = layers.Rescaling(
        residual_scale, name=f"{name}_residual_scale"
    )(residual)

    if use_residual:
        if x.shape[-1] != filters:
            shortcut = layers.Conv1D(
                filters, 1, padding="same", use_bias=False,
                name=f"{name}_shortcut_projection",
            )(shortcut)
            shortcut = layers.BatchNormalization(name=f"{name}_shortcut_bn")(shortcut)

        x = layers.Add(name=f"{name}_add")([shortcut, residual])
    else:
        # shortcut만 뺀 대조 실험입니다. 나머지 Inception 경로는 동일합니다.
        x = residual

    return layers.Activation("relu", name=f"{name}_out")(x)

## 8. B3 모델 만들기

Stem, 네 Inception-residual 블록, pooling, 분류 head는 B1·B2와 같습니다. 각 residual branch에만 CBAM을 추가합니다. `model.count_params()`를 B1·B2와 비교합니다.

손으로 계산한 대략적인 파라미터 수(BN 통계 포함)는 다음과 같습니다.

| 모델 | 추가 모듈 파라미터 | 전체 |
|---|---:|---:|
| B1 | – | 약 47,846 |
| B2 | SE 4개: 2,776 | 약 50,622 |
| B3 | CBAM 4개: 2,832 (SE와 같은 MLP + 시간 Conv 14개 × 4) | 약 50,678 |


In [ ]:
# B3 변경: use_se·se_ratio 대신 use_cbam·cbam_ratio·cbam_temporal_kernel 을 블록에 전달
def build_b3_model(
    use_residual=True, use_cbam=True, cbam_ratio=8,
    cbam_temporal_kernel=7, residual_scale=0.2,
):
    inputs = layers.Input(shape=(128, 9), name="sensor_window")

    x = layers.Conv1D(32, 5, padding="same", use_bias=False, name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("relu", name="stem_relu")(x)

    x = inception_residual_block(
        x, filters=32, bottleneck_filters=16,
        residual_scale=residual_scale, use_residual=use_residual, 
        use_cbam=use_cbam, cbam_ratio=cbam_ratio, cbam_temporal_kernel=cbam_temporal_kernel, name="ir32_1",
    )
    x = inception_residual_block(
        x, filters=32, bottleneck_filters=16,
        residual_scale=residual_scale, use_residual=use_residual, 
        use_cbam=use_cbam, cbam_ratio=cbam_ratio, cbam_temporal_kernel=cbam_temporal_kernel, name="ir32_2",
    )

    x = layers.MaxPooling1D(2, name="downsample_pool")(x)
    x = layers.Conv1D(64, 1, padding="same", use_bias=False, name="channel_up")(x)
    x = layers.BatchNormalization(name="channel_up_bn")(x)
    x = layers.Activation("relu", name="channel_up_relu")(x)

    x = inception_residual_block(
        x, filters=64, bottleneck_filters=24,
        residual_scale=residual_scale, use_residual=use_residual, 
        use_cbam=use_cbam, cbam_ratio=cbam_ratio, cbam_temporal_kernel=cbam_temporal_kernel, name="ir64_1",
    )
    x = inception_residual_block(
        x, filters=64, bottleneck_filters=24,
        residual_scale=residual_scale, use_residual=use_residual, 
        use_cbam=use_cbam, cbam_ratio=cbam_ratio, cbam_temporal_kernel=cbam_temporal_kernel, name="ir64_2",
    )

    x = layers.GlobalAveragePooling1D(name="global_average_pooling")(x)
    x = layers.Dropout(0.4, name="classifier_dropout")(x)
    outputs = layers.Dense(n_outputs, activation="softmax", name="activity")(x)

    return Model(inputs, outputs, name="UCI_HAR_B3_Inception_ResNet1D_CBAM")


model = build_b3_model(
    use_residual=USE_RESIDUAL,
    use_cbam=USE_CBAM,
    cbam_ratio=CBAM_RATIO,
    cbam_temporal_kernel=CBAM_TEMPORAL_KERNEL,
    residual_scale=RESIDUAL_SCALE,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
print(f"parameters: {model.count_params():,}")

## 9. `Concatenate`·`Add`·CBAM shape 확인

`Concatenate` 뒤에는 여러 branch의 채널이 모이고, projection을 거친 residual과 shortcut은 같은 shape이어야 합니다. 이 검사는 잘못된 `Add` 설계를 실행 전에 발견하기 위한 것입니다.

B3에서는 CBAM 전후 shape도 같아야 하고, 시간 attention 가중치는 `(batch, time, 1)`이어야 합니다. CBAM은 값을 조절하지만 시간 길이와 채널 수를 바꾸지 않습니다.


In [ ]:
for block_name in ["ir32_1", "ir32_2", "ir64_1", "ir64_2"]:
    merged_shape = model.get_layer(f"{block_name}_concatenate").output.shape
    residual_shape = model.get_layer(f"{block_name}_residual_bn").output.shape
    output_shape = model.get_layer(f"{block_name}_out").output.shape
    print(block_name, "concat:", merged_shape, "residual:", residual_shape, "out:", output_shape)

    if USE_RESIDUAL:
        add_layer = model.get_layer(f"{block_name}_add")
        left_shape, right_shape = [tensor.shape for tensor in add_layer.input]
        assert left_shape == right_shape, f"{block_name} Add shape 불일치"

    # B3 변경: SE 레이어 대신 CBAM 레이어 이름으로 shape 확인
    if USE_CBAM:
        cbam_input_shape = model.get_layer(f"{block_name}_residual_bn").output.shape
        cbam_output_shape = model.get_layer(f"{block_name}_cbam_temporal_scale").output.shape
        temporal_weight_shape = model.get_layer(f"{block_name}_cbam_temporal_conv").output.shape
        assert cbam_input_shape == cbam_output_shape, f"{block_name} CBAM shape 불일치"
        assert temporal_weight_shape[-1] == 1, f"{block_name} 시간 attention 채널은 1이어야 함"
        print(" " * 10, "CBAM in/out:", cbam_input_shape, "->", cbam_output_shape,
              "| temporal weight:", temporal_weight_shape)


## 10. 학습

기존 S0와 마찬가지로 최대 100에폭, batch size 64, Adam과 EarlyStopping을 사용합니다. 구조 외의 조건을 가능한 한 고정하고 학습 시간도 함께 기록합니다.

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=10,
        restore_best_weights=True, verbose=1,
    )
]

train_start = time.perf_counter()

history = model.fit(
    X_train,
    y_train_onehot,
    validation_data=(X_val, y_val_onehot),
    epochs=100,
    batch_size=64,
    shuffle=True,
    callbacks=callbacks,
    verbose=1,
)

training_seconds = time.perf_counter() - train_start
best_epoch = int(np.argmin(history.history["val_loss"])) + 1

print(f"training time: {training_seconds:.1f} sec")
print(f"best epoch by val_loss: {best_epoch}")

## 11. Validation 전체 평가

In [ ]:
class_names = [
    "WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS",
    "SITTING", "STANDING", "LAYING",
]

y_val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)
cm_val = confusion_matrix(y_val, y_val_pred, labels=range(n_outputs))

print(f"validation accuracy : {val_accuracy:.4f}")
print(f"validation macro F1 : {val_f1:.4f}")
print(classification_report(
    y_val, y_val_pred, target_names=class_names,
    digits=4, zero_division=0,
))
print("confusion matrix")
print(cm_val)

## 12. `SITTING`/`STANDING`과 피험자별 평가

전체 정확도가 좋아져도 기존 핵심 문제인 앉음↔섬 오류가 늘어날 수 있으므로 별도로 기록합니다.

In [ ]:
SIT, STAND = 3, 4

print("subject | all acc | SIT/STAND acc | SIT->STAND | STAND->SIT")
for subject in VAL_SUBJECTS:
    mask = s_val == subject
    ss_mask = mask & np.isin(y_val, [SIT, STAND])
    sit_to_stand = np.sum(mask & (y_val == SIT) & (y_val_pred == STAND))
    stand_to_sit = np.sum(mask & (y_val == STAND) & (y_val_pred == SIT))
    mark = " <- focus" if subject in FOCUS_SUBJECTS else ""

    print(
        f"{subject:7d} | {accuracy_score(y_val[mask], y_val_pred[mask]):.4f}  | "
        f"{accuracy_score(y_val[ss_mask], y_val_pred[ss_mask]):.4f}        | "
        f"{sit_to_stand:10d} | {stand_to_sit:10d}{mark}"
    )

focus_mask = np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])
other_mask = ~np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

focus_ss_acc = accuracy_score(y_val[focus_mask], y_val_pred[focus_mask])
other_ss_acc = accuracy_score(y_val[other_mask], y_val_pred[other_mask])
val_ss_err = int(cm_val[SIT, STAND] + cm_val[STAND, SIT])
val_dynamic_err = int(cm_val[0:3, 0:3].sum() - np.trace(cm_val[0:3, 0:3]))

## 13. 추론 시간 측정

GPU·CPU 종류에 따라 절대 시간은 달라집니다. 따라서 같은 Colab 런타임에서 다른 모델과 측정한 값끼리만 비교합니다.

In [ ]:
timing_batch = tf.convert_to_tensor(X_val[:256])

# 초기 그래프 준비와 장치 준비 시간을 제외하기 위한 warm-up입니다.
for _ in range(3):
    model(timing_batch, training=False).numpy()

repeats = 20
inference_start = time.perf_counter()
for _ in range(repeats):
    model(timing_batch, training=False).numpy()
inference_seconds = time.perf_counter() - inference_start

inference_ms_per_sample = (
    inference_seconds * 1000 / (repeats * len(timing_batch))
)
print(f"inference: {inference_ms_per_sample:.4f} ms/sample")

## 14. Gradient norm 관찰

고정된 validation mini-batch에서 loss를 한 번 계산하고 각 Conv1D kernel의 gradient norm을 출력합니다. 앞쪽 층의 gradient가 반복적으로 매우 작고 residual을 끈 실험보다도 작다면 gradient 전달 문제를 의심할 수 있습니다.

단, 한 시점의 gradient norm만으로 gradient 소실을 증명할 수는 없습니다. 동일 seed에서 `USE_RESIDUAL=True/False`를 비교하고 학습곡선·수렴 에폭·정확도를 함께 봅니다.

B3에서는 CBAM이 residual branch의 크기를 채널·시점별로 두 번 바꾸므로 B1·B2와 gradient norm이 달라질 수 있습니다. 이 차이를 곧바로 더 좋은 gradient 전달이라고 해석하지 않고 성능과 수렴 양상을 함께 봅니다.


In [ ]:
probe_x = tf.convert_to_tensor(X_val[:64])
probe_y = tf.convert_to_tensor(y_val_onehot[:64])

with tf.GradientTape() as tape:
    probe_prediction = model(probe_x, training=False)
    probe_loss = tf.reduce_mean(
        tf.keras.losses.categorical_crossentropy(probe_y, probe_prediction)
    )

gradients = tape.gradient(probe_loss, model.trainable_variables)
gradient_rows = []

for variable, gradient in zip(model.trainable_variables, gradients):
    variable_path = getattr(variable, "path", variable.name)
    if gradient is not None and "kernel" in variable_path:
        gradient_rows.append((variable_path, float(tf.norm(gradient))))

print("loss:", float(probe_loss))
print("\n앞쪽 kernel gradient norm")
for name, norm in gradient_rows[:6]:
    print(f"{name:55s} {norm:.8f}")

print("\n뒤쪽 kernel gradient norm")
for name, norm in gradient_rows[-6:]:
    print(f"{name:55s} {norm:.8f}")

## 15. 학습곡선

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} loss · seed {SEED}")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
plt.xlabel("epoch")
plt.title(f"{EXPERIMENT_NAME} accuracy · seed {SEED}")
plt.legend()

plt.tight_layout()
plt.show()

## 16. 결과 한 줄 요약

In [ ]:
print(
    f"{EXPERIMENT_NAME} | seed {SEED} | "
    f"val acc {val_accuracy:.4f} | val F1 {val_f1:.4f} | "
    f"focus SIT/STAND(5,17) {focus_ss_acc:.4f} | "
    f"other SIT/STAND(11,23,29) {other_ss_acc:.4f} | "
    f"SIT<->STAND errors {val_ss_err} | dynamic errors {val_dynamic_err} | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | "
    f"params {model.count_params():,} | train sec {training_seconds:.1f} | "
    f"inference ms/sample {inference_ms_per_sample:.4f}"
)

## 17. B1·B2·B3 결과 기록표

B1·B2·B3를 같은 seed끼리 비교합니다. 동적 오류(dynamic errors)와 SIT↔STAND 오류를 따로 기록해 시간 attention이 어느 쪽에 작용했는지 봅니다.

| 실험 | seed | Attention | Val Accuracy | Macro F1 | Focus SIT/STAND | SIT↔STAND | Dynamic errors | Params | Best epoch | Train sec | ms/sample |
|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| B1 | 0 | 없음 |  |  |  |  |  |  |  |  |  |
| B2 | 0 | SE |  |  |  |  |  |  |  |  |  |
| B3 | 0 | CBAM |  |  |  |  |  |  |  |  |  |
| B1 | 1 | 없음 |  |  |  |  |  |  |  |  |  |
| B2 | 1 | SE |  |  |  |  |  |  |  |  |  |
| B3 | 1 | CBAM |  |  |  |  |  |  |  |  |  |
| B1 | 2 | 없음 |  |  |  |  |  |  |  |  |  |
| B2 | 2 | SE |  |  |  |  |  |  |  |  |  |
| B3 | 2 | CBAM |  |  |  |  |  |  |  |  |  |

### 해석 기준

- 세 seed 평균과 표준편차를 보고, 같은 seed에서 `B3 − B1`, `B3 − B2` 방향이 반복되는지 확인합니다.
- 가설대로라면 B3의 이득은 **동적 오류 감소**에서 먼저 보이고, SIT↔STAND 오류는 B2와 비슷해야 합니다. 반대로 SIT/STAND만 바뀌었다면 시간 attention보다 채널 attention의 max pooling 경로 차이를 먼저 의심합니다.
- B3가 B2보다 좋지 않으면 "이 데이터에서는 시간 위치 재가중이 필요하지 않았다"는 근거가 됩니다. 이것도 의미 있는 결과입니다.
- B3의 추가 연산과 추론 시간에 비해 개선 폭이 충분한지도 확인합니다.
- CBAM이 좋아도 "원본 센서축이나 특정 동작 순간을 골랐다"고 결론 내리지 않습니다. 학습 특징의 채널·시점 재가중이 도움이 되었을 가능성까지만 말합니다.


## 18. 🔒 공식 test 평가

모든 후보를 같은 validation 조건으로 비교하고 최종 모델을 결정한 뒤 한 번만 실행합니다. 기본값에서는 오류를 발생시키지 않고 “건너뜀” 메시지만 출력하므로, Colab의 “모두 실행” 결과를 깨끗하게 저장할 수 있습니다.

In [ ]:
FINAL_TEST = False

if not FINAL_TEST:
    print(
        "공식 test를 건너뜁니다. validation으로 최종 모델을 정한 뒤 "
        "이 셀의 FINAL_TEST=True로 변경해 한 번만 실행하세요."
    )
else:
    y_test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)
    cm_test = confusion_matrix(y_test, y_test_pred, labels=range(n_outputs))

    print(f"test accuracy : {test_accuracy:.4f}")
    print(f"test macro F1 : {test_f1:.4f}")
    print(classification_report(
        y_test, y_test_pred, target_names=class_names,
        digits=4, zero_division=0,
    ))
    print("confusion matrix")
    print(cm_test)

    test_ss_err = int(cm_test[SIT, STAND] + cm_test[STAND, SIT])
    test_dynamic_err = int(cm_test[0:3, 0:3].sum() - np.trace(cm_test[0:3, 0:3]))

    print(
        f"{EXPERIMENT_NAME} TEST | seed {SEED} | acc {test_accuracy:.4f} | "
        f"macro F1 {test_f1:.4f} | SIT<->STAND {test_ss_err} | "
        f"dynamic errors {test_dynamic_err}"
    )

## 19. 참고 자료

- Szegedy et al., *Inception-v4, Inception-ResNet and the Impact of Residual Connections on Learning*, 2016
- He et al., *Deep Residual Learning for Image Recognition*, 2015
- Hu et al., *Squeeze-and-Excitation Networks*, 2017
- Woo et al., *CBAM: Convolutional Block Attention Module*, ECCV 2018
- Ismail Fawaz et al., *InceptionTime: Finding AlexNet for Time Series Classification*, 2019
- UCI Machine Learning Repository, *Human Activity Recognition Using Smartphones*